**Car Explorer – Vehicle Database & Cost Analysis**
A program that helps you edit our vehicle database, filter and search cars and create analysis.

**CRUD:**

In [ ]:
import csv

first_line = list()

data = {}

with open("cars.csv") as CSV_file:
    i = 0
    for row in CSV_file:
        formatted_row = row.strip('"').split('","')
        formatted_row[-1] = formatted_row[-1].rstrip('"\r\n')
        if i == 0:
            first_line = formatted_row
        else:
            data[i - 1] = {
                first_line[j]: formatted_row[j]
                for j in range(len(first_line))
            }

        i += 1

#Basic Get Car data by CAR_ID. CAR_ID ranges from 0-len(data)
def get_car(car_id):
    if car_id in data:
        return data[car_id]
    else:
        return None

def delete_car(car_id):
    if car_id in data:
        del data[car_id]
        return(f"Deleted {car_id} data")

def add_car(**kwargs):
    car = {}
    for items in first_line:
        if items in kwargs.keys():
            car[items] = kwargs[items]
        else:
            car[items] = ""

    new_id = max(data.keys(), default=-1) + 1 #create a never used ID with getting the biggest exisiting one and adding one to it. If empty list then 0.
    data[new_id] = car #add car to data to the new ID position.


    return(f"Added {car}")


#NOT YET WRITTEN WITH **kwargs
def update_car(car_id, **kwargs):
    if car_id not in data:
        return "Car not found"

    for attribute in kwargs:
        if attribute not in first_line:
            return f'Invalid attribute: {attribute}'

    for attribute, value in kwargs.items():
        data[car_id][attribute] = value

    return f"Car {car_id} updated successfully"



def save_cars(filename):
    with open(filename, "w") as file:
        file.write('"' + '","'.join(first_line) + '"\n')

        for car in data.values():
            row = [str(car.get(attribute, "")).replace('"', '""') for attribute in first_line]
            file.write('"' + '","'.join(row) + '"\n')

    return "Database saved successfully"



Search functions

In [20]:
def search_by_make(make):
    results = list()
    for car in data:
       if data[car]["Identification.Make"] == make:
           results.append(data[car])
    return results

def search_by_year(year: int):
     results = list()
     for car in data:
        try:
            modelyear = int(data[car]["Identification.Year"])
        except ValueError:
            modelyear = 0

        if modelyear == year:
             results.append(data[car])
     return results

def search_by_hp(horsepower: int):
     results = list()
     for car in data:
        try:
            hp = int(data[car]["Engine Information.Engine Statistics.Horsepower"])
        except ValueError:
            hp = 0

        if hp == horsepower:
             results.append(data[car])
     return results

def find_most_powerful():
    max = 0
    id_max = 0
    for id in data:
        if int(data[id]["Engine Information.Engine Statistics.Horsepower"]) > max:
            max = int(data[id]["Engine Information.Engine Statistics.Horsepower"])
            id_max = id

    return f'The most powerful is: {max}, carID: {id_max}'


def count_by_transmission():
    result = dict()
    for car in data.keys():
        trans = data[car]["Identification.Classification"]
        if trans not in result.keys():
            result[trans] = 1
        else:
            result[trans] += 1

    return result


def sort_by_horsepower():
    for i in range(len(data)-1):
        for j in range(len(data)-i-1):
            if data[j]["Engine Information.Engine Statistics.Horsepower"] > data[j+1]["Engine Information.Engine Statistics.Horsepower"]:
                data[j]["Engine Information.Engine Statistics.Horsepower"], data[j+1]["Engine Information.Engine Statistics.Horsepower"] = data[j+1]["Engine Information.Engine Statistics.Horsepower"], data[j]["Engine Information.Engine Statistics.Horsepower"]

    return data

def get_brands():
    brands = set()

    for car in data.values():
        brands.add(car["Identification.Make"])

    return sorted(brands)



['AMG', 'Acura', 'Aston Martin', 'Audi', 'BMW', 'BMW Motorrad', 'Bentley', 'Buick', 'Cadillac', 'Chevrolet', 'Chrysler', 'Chrysler Group LLC', 'Dodge', 'Ferrari', 'Ford', 'GMC', 'Grand Cherokee', 'Honda', 'Hyundai', 'Infiniti', 'Jaguar', 'Jeep', 'Kia', 'Lamborghini', 'Land Rover', 'Lexus', 'Lincoln', 'Lotus', 'MINI', 'Maserati', 'Maybach', 'Mazda', 'Mercedes', 'Mercedes-AMG', 'Mercedes-Benz', 'Mercury', 'Mitsubishi', 'Nissan', 'Porsche', 'Rolls-Royce', 'Saab', 'Scion', 'Subaru', 'Suzuki', 'Toyota', 'Volkswagen', 'Volvo']


**Statistical Functions**

In [ ]:
def avarage_horsepower(): #returns exact avarage horsepower (adds 0HP when no data provided)
    total_hp = 0
    for cars in data:
        try:
            hp = int(data[cars]["Engine Information.Engine Statistics.Horsepower"])
        except ValueError:
            hp = 0
        total_hp += hp

    return total_hp / len(data)

def mpg_to_l100km(mpg:float):
    if mpg <= 0:
        return None
    return 235.215/mpg

def annual_fuel_cost(car_id:int, yearly_km, fuel_price_per_liter:float, city_ratio:float=0.5):
    car = get_car(car_id)

    if car is None:
        return None

    if city_ratio > 1:
        return None
    elif city_ratio < 0:
        return None

    try:
        city_mpg = float(car["Fuel Information.City mpg"])
        highway_mpg = float(car["Fuel Information.Highway mpg"])
    except ValueError:
        return None

    city_consumption = mpg_to_l100km(city_mpg)
    highway_consumption = mpg_to_l100km(highway_mpg)

    if city_consumption is None or highway_consumption is None:
        return None

    avarage_consumption = (city_consumption * city_ratio + highway_consumption * (1 - city_ratio))

    annual_cost = (yearly_km /100) * avarage_consumption * fuel_price_per_liter

    return {round(annual_cost,2)}

def hybrid_percentage():
    hybrid_quantity = 0
    for car in data.keys():
        if data[car]["Engine Information.Hybrid"] == "True":
            hybrid_quantity += 1

    return round((hybrid_quantity / len(data)) * 100, 2)

print(hybrid_percentage())





**Available Car Brands**

In [ ]:
def get_brands():
    car_brands = []
    for brand in data:
        if data[brand]["Identification.Make"] not in car_brands:
            car_brands.append(data[brand]["Identification.Make"])
    return car_brands

print(get_brands())